In [0]:
#imports

In [0]:
###CONEXÕES

In [0]:
query = """
(
    SELECT 
        op.id,
        case 
            when op.id_enum_type_solicitation = 43 then op_mae.txt_policy_number
            else op.txt_policy_number
        end AS txt_policy_number,
        case 
            when op.id_enum_type_solicitation = 43 then op.txt_policy_number
            else null
        end as txt_endosso,
        cic.txt_cnpj as cnpj_seguradora,
        op.txt_documento_base,
        op.id_insurance_company
    FROM operational_policy op
    LEFT JOIN core_insurance_company cic 
        ON op.id_insurance_company = cic.id
    LEFT JOIN operational_policy op_mae
        ON op.id_initial_policy = op_mae.id
    WHERE op.source = 0 
        and op.dt_emission > '2025-01-01'
        and
        (
            op.vl_premium > 4000
            OR op.vl_insured_value > 1000000
        )
        AND op.dt_logical_delete IS NULL
        and op.id_insurance_company not in (1,2, 21)
        and op.txt_documento_base is not null
        AND op.dt_start_policy IS NOT NULL
        AND op.dt_end_policy IS NOT NULL
        and (op.has_refound = 0 or op.has_refound is null)
        and (op.refound_value is null or op.refound_value = 0)
        and (op.fl_cosseguro = 0 or op.fl_cosseguro is null)
        AND op.enum_endorsement_type <> 1
        AND NOW() BETWEEN op.dt_start_policy AND op.dt_end_policy
        AND
        (
            op.status IN (6, 10, 14, 18)
            OR
            (
                op.status = 17
                AND EXISTS (
                    SELECT 1
                    FROM operational_policy_renew opr
                    WHERE opr.id_policy_old = op.id
                      AND opr.status = 4
                )
            )
        )
) query_df


"""




In [0]:
df = spark.read.jdbc(
    url=jdbcUrl,
    table=query,
    properties=connectionProperties
)



In [0]:
df.createOrReplaceTempView("fato_operacional")
fato_operacional_df = spark.table("fato_operacional")

In [0]:
%skip
%sql
CREATE TABLE IF NOT EXISTS policy_document_read_control (
    policy_id               BIGINT,
    num_apolice             STRING,
    num_endosso             STRING,
    document_hash           STRING,
    valor_premio            DECIMAL(18,2),
    valor_is                DECIMAL(18,2),
    status                  STRING, -- PENDENTE , EM_PROCESSAMENTO , PROCESSADO, ERRO
    cnpj_seguradora          STRING,
    inicio_vigencia         TIMESTAMP,
    fim_vigencia            TIMESTAMP,
    dt_emissiao             TIMESTAMP,
    dt_primeiro_envio       TIMESTAMP,
    dt_ultima_tentativa     TIMESTAMP,
    reader_response_json     STRING,
    mensagem_erro            STRING,
    tentativas               INT
)
USING DELTA;


In [0]:
df_controle = spark.table("hive_metastore.default.policy_document_read_control")
df_controle.createOrReplaceTempView("view_control")

In [0]:

deltaTable = DeltaTable.forName(
    spark,
    "default.policy_document_read_control"
)

deltaTable.alias("target").merge(
    fato_operacional_df.alias("source"),
    "target.policy_id = source.id"
).whenNotMatchedInsert(
    values={
        "policy_id": "source.id",
        "num_apolice": "source.txt_policy_number",
        "num_endosso": "source.txt_endosso",
        "cnpj_seguradora": "source.cnpj_seguradora",
        "document_hash": "md5(concat(source.txt_policy_number, source.txt_endosso))",
        "status": "'PENDENTE'", 
        "origem_api": "'1'"
    }
).execute()

In [0]:
##microsoft conexão

In [0]:
###URL_PA

In [0]:
#header pa token


In [0]:
df_controle = spark.table("hive_metastore.default.policy_document_read_control")
df_controle.createOrReplaceTempView("view_control")

In [0]:
lote = """
    SELECT policy_id
    FROM view_control
    WHERE status = 'PENDENTE'
    order by 1
    LIMIT 10
"""
lote_df = spark.sql(lote)
lote_df.createOrReplaceTempView("lote")

#if lote_df.count() == 0:
#    print("Nada pendente")
#    dbutils.notebook.exit("OK")




In [0]:
spark.sql("""
MERGE INTO policy_document_read_control t
USING lote l
ON t.policy_id = l.policy_id
WHEN MATCHED AND t.status = 'PENDENTE' THEN
  UPDATE SET
    t.status = 'EM_PROCESSAMENTO',
    t.dt_primeiro_envio = CASE
        WHEN t.dt_primeiro_envio IS NULL THEN current_timestamp()
        ELSE t.dt_primeiro_envio
    END,
    t.dt_ultima_tentativa = CASE
        WHEN t.dt_primeiro_envio IS NOT NULL THEN current_timestamp()
        ELSE t.dt_ultima_tentativa
    END

""").show


In [0]:
df_controle = spark.table("hive_metastore.default.policy_document_read_control")
df_controle.createOrReplaceTempView("view_control")

In [0]:
hashes_df = spark.sql("""
    SELECT 
        vc.document_hash
    FROM view_control vc
    WHERE vc.status = 'EM_PROCESSAMENTO'
    ORDER BY vc.document_hash
    LIMIT 10
""")


hashes = [f"'{row.document_hash}'" for row in hashes_df.collect()]
hashes_str = ",".join(hashes)

df_si = spark.read.jdbc(
    url=jdbcUrl,
    table=f"""
    (
        SELECT
            txt_name,
            content,
            SUBSTRING(
            txt_extension,
            LOCATE('data:', txt_extension) + 5,
            LOCATE(';base64', txt_extension) - (LOCATE('data:', txt_extension) + 5))             as txt_extension
        FROM storage_image
        WHERE txt_name IN ({hashes_str})
    ) tmp
    """,
    properties=connectionProperties
)



In [0]:
def enviar_para_ia(txt_name, content, txt_extension):
    body = {
        "nome_arquivo": txt_name,
        "conteudo_arquivo": content,
        "tipo_conteudo": txt_extension
    }

    try:
        response = requests.post(
            url_pa,
            headers=headers_pa,
            json=body,
            timeout=60
        )

        return response.status_code, response.json()

    except Exception as e:
        return "EXCEPTION", str(e)


In [0]:

# Responde da API só para ter de exemplo. :)
 
#response_json = {
#    'status': '200',
#    'mensagem': 'a Leitura desta apólice foi concluída utilizando Inteligência Artificial',
#    'dados_leitura': {
#        'numero_apolice': ''apolice,
#        'numero_endosso': None,
#        'cnpj_seguradora': 'cnpj',
#        'data_inicio_vigencia': '2025-07-18',
#        'data_fim_vigencia': '2028-03-31',
#       'data_emissao': '2025-07-25',
#        'valor_premio': 6298.07,
#        'valor_is': 1203283.68
#    }
# }

In [0]:
for row in df_si.toLocalIterator():
    status_code, payload = enviar_para_ia(
        row.txt_name,
        row.content,
        row.txt_extension
    )

    payload_str = str(payload).replace("'", "''")
    dados_leitura = payload.get("dados_leitura") if payload else None

    status = "ERRO_DESCONHECIDO"

    if status_code == 200 and dados_leitura:
        status = "PROCESSADO"

        valor_is = dados_leitura.get("valor_is")
        valor_premio = dados_leitura.get("valor_premio")
        cnpj_seguradora = dados_leitura.get("cnpj_seguradora")
        inicio_vig = dados_leitura.get("data_inicio_vigencia")
        fim_vig = dados_leitura.get("data_fim_vigencia")
        dt_emissao = dados_leitura.get("data_emissao")

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            valor_is = {valor_is if valor_is is not None else 'NULL'},
            valor_premio = {valor_premio if valor_premio is not None else 'NULL'},
            inicio_vigencia = {f"timestamp('{inicio_vig}')" if inicio_vig else 'NULL'},
            fim_vigencia = {f"timestamp('{fim_vig}')" if fim_vig else 'NULL'},
            dt_emissiao = {f"timestamp('{dt_emissao}')" if dt_emissao else 'NULL'},
            reader_response_json = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    elif status_code == 200 and not dados_leitura:
        status = "SUCESSO_SEM_DADOS"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            reader_response_json = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    elif status_code in (500, 502):
        status = "ERRO_TEMP"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            mensagem_erro = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    elif status_code in (400, 422):
        status = "DOC_INVALIDO"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            mensagem_erro = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    else:
        status = "ERRO_DESCONHECIDO"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            mensagem_erro = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)


In [0]:
df_resultado = (
    spark.table("hive_metastore.default.policy_document_read_control")
         .filter("status = 'PROCESSADO'")

)





In [0]:
df_sel = df_resultado.select(
    regexp_replace(col("num_apolice").cast("string"), r"\.0$", "").alias("num_apolice"),
    
    when(
        col("num_endosso").isNotNull(),
        regexp_replace(col("num_endosso").cast("string"), r"\.0$", "")
    ).otherwise(None).alias("num_endosso"),
    
    col("cnpj_seguradora").cast("string"),
    col("dt_emissiao"),
    col("inicio_vigencia"),
    col("fim_vigencia"),
    col("valor_premio").cast("double"),
    col("valor_is").cast("double")
)




In [0]:
pdf = df_sel.toPandas()

pdf["num_endosso"] = (
    pdf["num_endosso"]
        .fillna("")
)

for c in ["dt_emissiao", "inicio_vigencia", "fim_vigencia"]:
    pdf[c] = pd.to_datetime(pdf[c]).dt.strftime("%d-%m-%Y")

buffer = BytesIO()

with pd.ExcelWriter(buffer, engine="xlsxwriter") as writer:
    pdf.to_excel(writer, sheet_name="Planilha1", index=False)

agora = datetime.now()
timestamp = agora.strftime("%d-%m-%Y_%H-%M")

nome_arquivo = f"template_comparacao_{timestamp}.xlsx"

output_path = f"/dbfs/FileStore/tables/{nome_arquivo}"

with open(output_path, "wb") as f:
    f.write(buffer.getvalue())



In [0]:
tmp_output_path = "/tmp/MODELO_TEMPLATE_COMPARACAO_PREENCHIDO.xlsx"
with pd.ExcelWriter(tmp_output_path, engine="xlsxwriter") as writer:
    pdf.to_excel(writer, sheet_name="Planilha1", index=False)


In [0]:
template_dbfs = "dbfs:/FileStore/tables/MODELO_TEMPLATE_COMPARACAO.xlsx"
template_local = "/tmp/MODELO_TEMPLATE_COMPARACAO.xlsx"

dbutils.fs.cp(template_dbfs, f"file:{template_local}")


In [0]:
wb = load_workbook(template_local)
ws = wb["Planilha1"]


In [0]:
sheet_name = "Informacoes"

if sheet_name in wb.sheetnames:
    ws_info = wb[sheet_name]
    ws_info.delete_rows(1, ws_info.max_row) 
else:
    ws_info = wb.create_sheet(sheet_name)

In [0]:
headers = ["Versao", "Tipo", "Template", "Dt_Modificacao"]

ws_info.append(headers)


In [0]:
values = [
    "v2",
    "template-comparacao-generica",
    "templates-comparacao-avita",
    datetime(2024, 6, 26)
]

ws_info.append(values)

# formato da data
ws_info["D2"].number_format = "DD/MM/YYYY"


In [0]:
header_row = 1
header_map = {
    cell.value: cell.column
    for cell in ws[header_row]
    if cell.value
}


In [0]:
mapping = {
    "Número da Apólice*": "num_apolice",
    "Número do Endosso": "num_endosso",
    "Data de Emissão*": "dt_emissiao",
    "Inicio de Vigência*": "inicio_vigencia",
    "Fim de Vigência*": "fim_vigencia",
    "Valor do Prêmio*": "valor_premio",
    "Importância Segurada*": "valor_is",
    "Cnpj Seguradora*":"cnpj_seguradora",
}


In [0]:
def parse_iso(value):
    if value is None:
        return None
    if isinstance(value, datetime):
        return value
    return datetime.fromisoformat(str(value).split("+")[0])


In [0]:
start_row = 2

TEXT_COLS = ["num_apolice", "num_endosso", "cnpj_seguradora"]

for i, row in pdf.iterrows():
    excel_row = start_row + i

    for excel_col, view_col in mapping.items():
        col_idx = header_map.get(excel_col)
        if not col_idx:
            continue

        value = row[view_col]
        cell = ws.cell(row=excel_row, column=col_idx)

        if view_col in TEXT_COLS:
            if pd.isna(value):
                cell.value = ""
            else:
                cell.value = str(value)
            cell.number_format = "@"

        elif view_col in ["dt_emissiao", "inicio_vigencia", "fim_vigencia"]:
            if not pd.isna(value):
                cell.value = value
                cell.number_format = "DD-MM-YYYY"
            else:
                cell.value = None

        elif view_col in ["valor_is", "valor_premio"]:
            if not pd.isna(value):
                cell.value = float(value)
                cell.number_format = "#,##0.00"
            else:
                cell.value = None

        else:
            cell.value = None


In [0]:
output_local = "/tmp/MODELO_TEMPLATE_COMPARACAO_PREENCHIDO.xlsx"
wb.save(output_local)


In [0]:
###Conexões



In [0]:
#GET TOKEN


In [0]:
file_name = os.path.basename(LOCAL_FILE_PATH)

upload_url = (
    url
)

with open(LOCAL_FILE_PATH, "rb") as f:
    response = requests.put(
        ###upload_url
        
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/octet-stream"
        },
        data=f
    )

print("Status:", response.status_code)
print(response.text)
